# Tokenization

These exercises build on last week's lecture. First, you'll build a tokenizer yourself. Then, you'll load and examine a pretrained tokenizer, getting used to how these more complex tokenizers work. Finally, you'll examine the downstream effects of tokenization using GPT-2. 

## Exercise 1: Write and Train a Simple Tokenizer

You'll be designing a basic tokenizer. This won't have to work with any model downstream, and is just an exercise to reinforce what a tokenizer does. You have complete control over design decisions, subject to the following constraints:
1. You train your tokenizer on the provided dataset (Python code snippets).
2. It cannot throw errors when tokenizing any (unseen) words. 
3. It must do more than just split on whitespace.

After you've designed and trained your tokenizer, test it on the provided sequences. 

In [1]:
from datasets import load_dataset

In [3]:
# Take first 10k of train set
code_dataset = load_dataset("flytech/python-codes-25k", split="train[:10000]")

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

In [7]:
print(code_dataset)

Dataset({
    features: ['output', 'instruction', 'input', 'text'],
    num_rows: 10000
})


In [ ]:
# Fill this in
class SimpleTokenizer:
    def __init__(self):
        self.idx2token = {} 
        self.token2idx = {} 
        pass

    def train(self, corpus):
        pass

    def tokenize(self, text):
        """Turn a string into a list of IDs corresponding to tokens."""
        pass

    def untokenize(self, tokens):
        """Turn a list of IDs into a list of tokens"""
        pass

    def get_vocab_size(self):
        return len(self.idx2token)

In [ ]:
# Train your tokenizer on the "output" column of the dataset
simple_tokenizer = SimpleTokenizer(...)
simple_tokenizer.train(...)


In [ ]:
regular_text = "Ողջույն! Γεια σας! Përshëndetje! ሰላም!"

code_text = '''
def my_function():
    print("Hello world!")
'''

text_tokenized = simple_tokenizer.tokenize(regular_text)
code_tokenized = simple_tokenizer.tokenize(code_text)

# See how your tokenizer performed
print(simple_tokenizer.untokenize(text_tokenized))
print(simple_tokenizer.untokenize(code_tokenized))

## Exercise 2: Examine a Pretrained Tokenizer

### Questions to answer with your investigation:
1. What kind of model is this tokenizer used with?
2. What special tokens does it have? What do they do?
3. Tokenize some code. What does it look like? 
4. Now tokenize some regular text of your choosing. How does it look?  

In [9]:
from transformers import AutoTokenizer

In [ ]:
model_name = "Pick a model to analyze"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Write whatever code you need to answer the questions above

## Exercise 3: Downstream Effects

Now we'll tie tokenization back into language modeling. 

In [19]:
import torch
from transformers import AutoModelForCausalLM

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Remember that the tokenizer used for a model like GPT-2 splits words into *subwords*. Thus, when the model makes predictions, it is not always predicting the next word, but the next subword unit. 

Your task for this section: 
1. Use the tokenizer to identify some interesting subwords and/or words that are split into several subwords. You can either search the vocabulary or use trial and error by tokenizing words/sequences. Think about what kinds of words are likely to be split into multiple subwords.
2. Write a few prompts where you think these subwords will be likely continuations. 
3. Analyze logits to see if you were correct.

In [24]:
# Your code here. 
# Below is a reminder for how to get logits for a single sequence
with torch.no_grad():
    logits = gpt2(**gpt2_tokenizer("Hello!", return_tensors="pt")).logits
print(logits.shape)

torch.Size([1, 2, 50257])
